# 1 - Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [3]:
import pandas as pd
import numpy as np

In [4]:
from src.utils import config, io, countries

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


# 1 - Build Catalogs

In [ ]:
from extraction import world_bank

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
client = world_bank.DBnomicsClient()

## 1.1 Dataset Catalog

In [ ]:
### dbnomics_lib_client.py

import pandas as pd
import dbnomics

def format_api_link(provider: str, dataset: str, indicator: str) -> str:
    """get api link for the indicator.

    Args:
        provider (string): name of the provider of the indicator.
        dataset (string): name of the dataset of the indicator.
        indicator (string): name of the indicator.

    Returns:
        string: formated api link, to get indicator df from dbnomics api.
    """

    link_p1 = 'https://api.db.nomics.world/v22/series/'
    link_p3 = '%22%5D%7D&observations=1'

    if provider == 'WB':
        link_p2 = '?dimensions=%7B%22indicator%22%3A%5B%22'
    elif provider == 'IMF':
        link_p2 = '?dimensions=%7B%22INDICATOR%22%3A%5B%22'
        # link_p3 = link_p3 + '&q=imf'
    else:
        return ''

    api_link = link_p1 + provider + '/' + dataset + link_p2 + indicator + link_p3
    return api_link

def fetch_indicator(provider: str, dataset: str, indicator: str) -> pd.DataFrame:
    """get unformated indicator df from dbnomics api.
    Args:
        provider (string): name of the provider of the indicator.
        dataset (string): name of the dataset of the indicator.
        indicator (string): name of the indicator.
    Returns:
        DataFrame: indicator df from dbnomics api.
    """

    api_link = format_api_link(provider, dataset, indicator)
    try:
        df_indicator = dbnomics.fetch_series_by_api_link(
            api_link,
            max_nb_series=600
        )
    except Exception:
        print('Error Fetching Series for indicator', indicator)
        df_indicator = pd.DataFrame()

    return df_indicator

### 1.1.1 - Build Provider Datasets Catalog (Discontinued, Now Hand Picked List)

In [270]:
from src.metadata import build_provider_dataset_catalog

In [282]:
providers = [
    'IMF', 'WB', 'OECD', 'BIS', 'UNCTAD', 'FAO', 'ILO', 'FH'
]

In [283]:
datasets_df = build_provider_dataset_catalog.get_all_datasets_df(client, providers)
datasets_df

,index,Provider Id,Provider Name,Provider Region,Dataset Id,Dataset Name,Subgroup 1 Id,Subgroup 1 Name,Subgroup 2 Id,Subgroup 2 Name
0,0,IMF,International Monetary Fund,World,AFRREO,Sub-Saharan Africa Regional Economic Outlook (...,NaN,NaN,NaN,NaN
1,1,IMF,International Monetary Fund,World,APDREO,Asia and Pacific Regional Economic Outlook (AP...,NaN,NaN,NaN,NaN
2,2,IMF,International Monetary Fund,World,BOP,Balance of Payments (BOP),NaN,NaN,NaN,NaN
3,3,IMF,International Monetary Fund,World,BOPAGG,"Balance of Payments (BOP), World and Regional ...",NaN,NaN,NaN,NaN
4,4,IMF,International Monetary Fund,World,CDIS,Coordinated Direct Investment Survey (CDIS),NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2106,2562,ILO,International Labour Organization,World,WBL_TWBL_SEX_EDU_WBL_NB,"Work-based learners by sex, education and type...",WBL,Work-based learning,NaN,NaN
2107,2563,ILO,International Labour Organization,World,WBL_XVET_SEX_EDU_NB,Working-age population with vocational educati...,WBL,Work-based learning,NaN,NaN
2108,2564,ILO,International Labour Organization,World,WBL_3WBL_SEX_WBL_RT,Youth participation rate in work-based learnin...,WBL,Work-based learning,NaN,NaN
2109,2565,FH,Freedom House,World,FIW,"All data, Freedom in the world since 2013",NaN,NaN,NaN,NaN


In [284]:
datasets_df['Provider Id'].value_counts()

Provider Id
ILO       1071
OECD       758
IMF        106
FAO         85
UNCTAD      62
WB          14
BIS         13
FH           2
Name: count, dtype: int64

In [285]:
build_provider_dataset_catalog.TOTAL_AVOIDED

4550

In [ ]:
# io.save_csv(datasets_df, config.METADATA_DIR / 'provider_datasets.csv')

### 1.1.2 Prune Provider Datasets

In [7]:
datasets_df = io.load_csv(config.METADATA_DIR / 'provider_datasets_selected.csv', index_col=0)
datasets_df

,Provider Id,Provider Name,Provider Region,Dataset Id,Dataset Name,Subgroup 1 Id,Subgroup 1 Name,Subgroup 2 Id,Subgroup 2 Name,Country Param Name,Country Param Format,Frequency Param Name,Frequency Annual,Other Param Name
0,IMF,International Monetary Fund,World,BOP,Balance of Payments (BOP),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
2,IMF,International Monetary Fund,World,CPI,Consumer Price Index (CPI),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
3,IMF,International Monetary Fund,World,FDI,Financial Development Index,NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
4,IMF,International Monetary Fund,World,FM,Fiscal Monitor (FM),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
5,IMF,International Monetary Fund,World,FSI,Financial Soundness Indicators (FSIs),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
6,IMF,International Monetary Fund,World,GENDER_EQUALITY,Gender Equality,NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
7,IMF,International Monetary Fund,World,GFSMAB,"Government Finance Statistics (GFS), Main Aggr...",NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,CLASSIFICATION/REF_SECTOR/UNIT_MEASURE
8,IMF,International Monetary Fund,World,HPDD,Historical Public Debt (HPDD),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
9,IMF,International Monetary Fund,World,IFS,International Financial Statistics (IFS),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
10,IMF,International Monetary Fund,World,IRFCL,International Reserves and Foreign Currency Li...,NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR/REF_SECTOR


In [30]:
providers = datasets_df['Provider Id'].unique()

dataset_params = {}
for i, dataset_row in datasets_df.iterrows():
    provider_id = dataset_row['Provider Id']
    dataset_id = dataset_row['Dataset Id']
    print('Provider:', provider_id, 'Dataset:', dataset_id)

    if provider_id not in dataset_params.keys():
        dataset_params[provider_id] = {dataset_id: {}}
    else:
        dataset_params[provider_id][dataset_id] = {}

    try:
        dataset_dict = client.list_series(provider_id, dataset_id, limit=1)
        dataset_dim_dict = dataset_dict['dataset']['dimensions_values_labels']
        dataset_dim_dict.pop(dataset_row['Country Param Name'])
        dataset_dim_dict.pop(dataset_row['Frequency Param Name'])

        dataset_params[provider_id][dataset_id] = dataset_dim_dict

    except SyntaxError:
            print('Error querying dataset:', provider_id, dataset_id)

Provider: IMF Dataset: BOP
Provider: IMF Dataset: CPI
Provider: IMF Dataset: FDI
Provider: IMF Dataset: FM
Provider: IMF Dataset: FSI
Provider: IMF Dataset: GENDER_EQUALITY
Provider: IMF Dataset: GFSMAB
Provider: IMF Dataset: HPDD
Provider: IMF Dataset: IFS
Provider: IMF Dataset: IRFCL
Provider: IMF Dataset: MFS
Provider: IMF Dataset: PGCS
Provider: IMF Dataset: PSBSFAD
Provider: IMF Dataset: WoRLD
Provider: WB Dataset: DBS
Provider: WB Dataset: ESY
Provider: WB Dataset: FDX
Provider: WB Dataset: GEM
Provider: WB Dataset: GEP
Provider: WB Dataset: JOB
Provider: WB Dataset: WDI
Provider: WB Dataset: WGI
Provider: BIS Dataset: WS_GLI
Provider: BIS Dataset: WS_CREDIT_GAP
Provider: UNCTAD Dataset: MTTASA
Provider: UNCTAD Dataset: MTBA
Provider: UNCTAD Dataset: MTTGRA
Provider: UNCTAD Dataset: MPCADIOEAIA
Provider: UNCTAD Dataset: GASBTBIA
Provider: UNCTAD Dataset: GASBTOIA
Provider: UNCTAD Dataset: FDIIAOFASA
Provider: UNCTAD Dataset: GDPTAPCCAC2PA
Provider: UNCTAD Dataset: RGDPTAPCGRA
Pro

In [31]:
new_d = {}
for provider_id, provider_d in dataset_params.items():
    new_d[provider_id] = {}
    for dataset_id, dataset_d in provider_d.items():
        new_d[provider_id][dataset_id] = {}
        for param, param_d in dataset_d.items():
            new_d[provider_id][dataset_id][param] = {}

            new_param_d = {}
            for k, v in param_d.items():
                if (k[-4:] == '_EUR' or k[-4:] == '_XDC') and (k[:-4] + '_USD') in param_d.keys():
                    pass
                else:
                    new_param_d[k] = v

            new_d[provider_id][dataset_id][param] = new_param_d

In [32]:
new_d

{'IMF': {'BOP': {'INDICATOR': {'BACK_BP6_USD': 'Net Lending (+) / Net Borrowing (-) (Balance from Current and Capital Account), US Dollars',
    'BAXEF_BP6_USD': 'Supplementary Items, Arrears not in Exceptional Financing, US Dollars',
    'BCAXF_BP6_USD': 'Supplementary Items, Current Account, Net (Excluding Exceptional Financing), US Dollars',
    'BCA_BP6_SPE_USD': 'Current Account, Total, Net, Special Purpose Entities, US Dollar',
    'BCA_BP6_USD': 'Current Account, Total, Net, US Dollars',
    'BEFDDAAI_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Original Interest/Coupon, US Dollars',
    'BEFDDAAPI_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Penalty Interest, US Dollars',
    'BEFDDAAP_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Principal, US Dollars',
    'BEFDDAA_BP6_USD': 'Exceptional Financing, Direct Investment, Debt I

In [34]:
import json

json_str = json.dumps(new_d, indent=4)
with open(config.METADATA_DIR / 'datasets_dimensions.json', 'w') as f:
    f.write(json_str)

In [37]:
count_dataset_d = {}
count_d = {}
for provider_id, provider_d in new_d.items():
    count_d[provider_id] = {}
    count_dataset_d[provider_id] = {}
    for dataset_id, dataset_d in provider_d.items():
        count_d[provider_id][dataset_id] = {}
        count_dataset_d[provider_id][dataset_id] = {}
        v = 1
        for param, param_d in dataset_d.items():
            v = v * len(new_d[provider_id][dataset_id][param])
        
        count_dataset_d[provider_id][dataset_id] = v

In [36]:
count_d

{'IMF': {'BOP': {'INDICATOR': 2218},
  'CPI': {'INDICATOR': 178},
  'FDI': {'INDICATOR': 9},
  'FM': {'INDICATOR': 9},
  'FSI': {'INDICATOR': 959},
  'GENDER_EQUALITY': {'INDICATOR': 3},
  'GFSMAB': {'CLASSIFICATION': 35, 'REF_SECTOR': 8, 'UNIT_MEASURE': 2},
  'HPDD': {'INDICATOR': 1},
  'IFS': {'INDICATOR': 1301},
  'IRFCL': {'INDICATOR': 295, 'REF_SECTOR': 5},
  'MFS': {'INDICATOR': 575},
  'PGCS': {'INDICATOR': 15},
  'PSBSFAD': {'INDICATOR': 1235},
  'WoRLD': {'INDICATOR': 15}},
 'WB': {'DBS': {'indicator': 194},
  'ESY': {'indicator': 115},
  'FDX': {'indicator': 1226},
  'GEM': {'indicator': 36},
  'GEP': {'indicator': 1},
  'JOB': {'indicator': 166},
  'WDI': {'indicator': 1492},
  'WGI': {'indicator': 36}},
 'BIS': {'WS_GLI': {'BORROWERS_SECTOR': 20,
   'CURR_DENOM': 324,
   'LENDERS_SECTOR': 20,
   'L_INSTR': 23,
   'L_POS_TYPE': 8,
   'UNIT_MEASURE': 1088},
  'WS_CREDIT_GAP': {'CG_DTYPE': 3, 'TC_BORROWERS': 5, 'TC_LENDERS': 20}},
 'UNCTAD': {'MTTASA': {'flow': 2, 'measure': 2

In [38]:
count_dataset_d

{'IMF': {'BOP': 2218,
  'CPI': 178,
  'FDI': 9,
  'FM': 9,
  'FSI': 959,
  'GENDER_EQUALITY': 3,
  'GFSMAB': 560,
  'HPDD': 1,
  'IFS': 1301,
  'IRFCL': 1475,
  'MFS': 575,
  'PGCS': 15,
  'PSBSFAD': 1235,
  'WoRLD': 15},
 'WB': {'DBS': 194,
  'ESY': 115,
  'FDX': 1226,
  'GEM': 36,
  'GEP': 1,
  'JOB': 166,
  'WDI': 1492,
  'WGI': 36},
 'BIS': {'WS_GLI': 25944883200, 'WS_CREDIT_GAP': 300},
 'UNCTAD': {'MTTASA': 4,
  'MTBA': 2,
  'MTTGRA': 2,
  'MPCADIOEAIA': 6,
  'GASBTBIA': 8,
  'GASBTOIA': 16,
  'FDIIAOFASA': 20,
  'GDPTAPCCAC2PA': 4,
  'RGDPTAPCGRA': 2,
  'NGTAPCA': 2,
  'TAUPA': 2,
  'TPGRA': 1},
 'ILO': {'EAR_GGAP_OCU_RT': 4350,
  'SDG_1041_NOC_RT': 199,
  'EES_TEES_ECO_OCU_NB': 386512,
  'PSE_TPSE_GOV_NB': 2040,
  'GDP_205U_NOC_NB': 274,
  'SDG_B821_NOC_RT': 199,
  'INJ_FATL_ECO_NB': 12505}}

In [48]:
with open(config.METADATA_DIR / 'datasets_dimensions.json', 'r') as f:
    d = json.load(f)

print(d)

{'IMF': {'BOP': {'INDICATOR': {'BACK_BP6_USD': 'Net Lending (+) / Net Borrowing (-) (Balance from Current and Capital Account), US Dollars', 'BAXEF_BP6_USD': 'Supplementary Items, Arrears not in Exceptional Financing, US Dollars', 'BCAXF_BP6_USD': 'Supplementary Items, Current Account, Net (Excluding Exceptional Financing), US Dollars', 'BCA_BP6_SPE_USD': 'Current Account, Total, Net, Special Purpose Entities, US Dollar', 'BCA_BP6_USD': 'Current Account, Total, Net, US Dollars', 'BEFDDAAI_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Original Interest/Coupon, US Dollars', 'BEFDDAAPI_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Penalty Interest, US Dollars', 'BEFDDAAP_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Principal, US Dollars', 'BEFDDAA_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arr

In [51]:
d['BIS']['WS_GLI']

{'BORROWERS_SECTOR': {'A': 'All sectors',
  'B': 'Banks, total',
  'G': 'General government',
  'H': 'Households and NPISHs',
  'N': 'Non-banks, total'},
 'CURR_DENOM': {'USD': 'US dollar'},
 'LENDERS_SECTOR': {'A': 'All sectors',
  'B': 'Banks, total',
  'G': 'General government',
  'H': 'Households and NPISHs',
  'N': 'Non-banks, total'},
 'L_INSTR': {'A': 'All instruments',
  'B': 'Credit (loans & debt securities)',
  'D': 'Debt securities',
  'E': 'Risk-weighted assets',
  'G': 'Loans and deposits',
  'X': 'Equity instruments',
  'Y': 'Residual instrument'},
 'L_POS_TYPE': {'A': 'All'},
 'UNIT_MEASURE': {'770': 'Percentage of GDP'}}

In [53]:
count_dataset_d = {}
count_provider_d = {}
count_d = {}
for provider_id, provider_d in d.items():
    count_d[provider_id] = {}
    count_dataset_d[provider_id] = {}
    v_provider = 0
    for dataset_id, dataset_d in provider_d.items():
        count_d[provider_id][dataset_id] = {}
        count_dataset_d[provider_id][dataset_id] = {}
        v_dataset = 1
        for param, param_d in dataset_d.items():
            v_dataset = v_dataset * len(param_d)
        
        count_dataset_d[provider_id][dataset_id] = v_dataset
        v_provider = v_provider + v_dataset
    count_provider_d[provider_id] = v_provider

count_provider_d

{'IMF': 8547, 'WB': 3266, 'BIS': 250, 'UNCTAD': 69}

In [54]:
count_dataset_d

{'IMF': {'BOP': 2218,
  'CPI': 178,
  'FDI': 9,
  'FM': 9,
  'FSI': 959,
  'GENDER_EQUALITY': 3,
  'GFSMAB': 560,
  'HPDD': 1,
  'IFS': 1301,
  'IRFCL': 1475,
  'MFS': 575,
  'PGCS': 9,
  'PSBSFAD': 1235,
  'WoRLD': 15},
 'WB': {'DBS': 194,
  'ESY': 115,
  'FDX': 1226,
  'GEM': 36,
  'GEP': 1,
  'JOB': 166,
  'WDI': 1492,
  'WGI': 36},
 'BIS': {'WS_GLI': 175, 'WS_CREDIT_GAP': 75},
 'UNCTAD': {'MTTASA': 4,
  'MTBA': 2,
  'MTTGRA': 2,
  'MPCADIOEAIA': 6,
  'GASBTBIA': 8,
  'GASBTOIA': 16,
  'FDIIAOFASA': 20,
  'GDPTAPCCAC2PA': 4,
  'RGDPTAPCGRA': 2,
  'NGTAPCA': 2,
  'TAUPA': 2,
  'TPGRA': 1}}

In [756]:
d = client.list_series('ILO', 'LAP_2FTM_NOC_RT', limit=1)
d

{'_meta': {'args': {'align_periods': False,
   'dataset_code': 'LAP_2FTM_NOC_RT',
   'dimensions': {},
   'facets': False,
   'format': 'json',
   'limit': 1,
   'metadata': True,
   'observations': False,
   'offset': 0,
   'provider_code': 'ILO',
   'q': '',
   'series_code': None},
  'version': '22.1.17'},
 'dataset': {'attributes_labels': {'OBSV_STATUS': 'Observation status'},
  'attributes_values_labels': {'OBSV_STATUS': {'A': 'Adjusted',
    'B': 'Break in series',
    'C': 'Confidential',
    'E': 'Estimate',
    'I': 'Imputation',
    'M': 'Model-based extrapolation',
    'P': 'Provisional',
    'R': 'Real value',
    'S': 'Not significant',
    'U': 'Unreliable'}},
  'code': 'LAP_2FTM_NOC_RT',
  'dimensions_codes_order': ['ref_area', 'source', 'frequency'],
  'dimensions_labels': {'frequency': 'Frequency',
   'ref_area': 'Reference area',
   'source': 'Source'},
  'dimensions_values_labels': {'frequency': {'A': 'Annual'},
   'ref_area': {'X01': 'World',
    'X02': 'World: Low 

In [752]:
d.keys()

dict_keys(['_meta', 'dataset', 'errors', 'provider', 'series'])

In [753]:
d['dataset'].keys()

dict_keys(['attributes_labels', 'attributes_values_labels', 'code', 'dimensions_codes_order', 'dimensions_labels', 'dimensions_values_labels', 'dir_hash', 'indexed_at', 'name', 'nb_series', 'provider_code', 'provider_name'])

In [757]:
d['dataset']['dimensions_values_labels'].keys()

dict_keys(['frequency', 'ref_area', 'source'])

In [758]:
d['dataset']['dimensions_values_labels']['ref_area']

{'X01': 'World',
 'X02': 'World: Low income',
 'X03': 'World: Lower-middle income',
 'X04': 'World: Upper-middle income',
 'X05': 'World: High income',
 'X06': 'Africa',
 'X07': 'Africa: Low income',
 'X08': 'Africa: Lower-middle income',
 'X09': 'Africa: Upper-middle income',
 'X10': 'Northern Africa',
 'X11': 'Northern Africa: Lower-middle income',
 'X13': 'Sub-Saharan Africa',
 'X14': 'Sub-Saharan Africa: Low income',
 'X15': 'Sub-Saharan Africa: Lower-middle income',
 'X16': 'Sub-Saharan Africa: Upper-middle income',
 'X17': 'Central Africa',
 'X18': 'Eastern Africa',
 'X19': 'Southern Africa',
 'X20': 'Western Africa',
 'X21': 'Americas',
 'X23': 'Americas: Lower-middle income',
 'X24': 'Americas: Upper-middle income',
 'X25': 'Americas: High income',
 'X26': 'Latin America and the Caribbean',
 'X28': 'Latin America and the Caribbean: Lower-middle income',
 'X29': 'Latin America and the Caribbean: Upper-middle income',
 'X30': 'Latin America and the Caribbean: High income',
 'X31'

In [ ]:
BORROWERS_CTY/DSR_BORROWERS/FREQ

In [499]:
d['series'].keys()

dict_keys(['docs', 'limit', 'num_found', 'offset'])

In [659]:
d['series']['docs'][0]

{'dataset_code': 'WS_TC',
 'dataset_name': 'BIS long series on total credit',
 'dimensions': {'BORROWERS_CTY': '4T',
  'FREQ': 'Q',
  'TC_ADJUST': 'A',
  'TC_BORROWERS': 'C',
  'TC_LENDERS': 'A',
  'UNIT_TYPE': '770',
  'VALUATION': 'M'},
 'indexed_at': '2025-06-20T14:22:50.994Z',
 'provider_code': 'BIS',
 'series_code': 'Q.4T.C.A.M.770.A',
 'series_name': 'Quarterly – Emerging market economies (aggregate) – Non financial sector – All sectors – Market value – Percentage of GDP – Adjusted for breaks'}

In [656]:
import requests

url = "https://api.db.nomics.world/v22" + "/series/WB/ESY/?"

r = requests.get(url)
r.raise_for_status()
r.json()

{'_meta': {'args': {'align_periods': False,
   'dataset_code': 'ESY',
   'dimensions': {},
   'facets': False,
   'format': 'json',
   'limit': 1000,
   'metadata': True,
   'observations': False,
   'offset': 0,
   'provider_code': 'WB',
   'q': '',
   'series_code': None},
  'version': '22.1.17'},
 'dataset': {'code': 'ESY',
  'dimensions_codes_order': ['frequency', 'indicator', 'country'],
  'dimensions_values_labels': {'country': {'AFG': 'Afghanistan',
    'AGO': 'Angola',
    'ALB': 'Albania',
    'ARG': 'Argentina',
    'ARM': 'Armenia',
    'ATG': 'Antigua and Barbuda',
    'AZE': 'Azerbaijan',
    'BDI': 'Burundi',
    'BEL': 'Belgium',
    'BEN': 'Benin',
    'BFA': 'Burkina Faso',
    'BGD': 'Bangladesh',
    'BGR': 'Bulgaria',
    'BHS': 'Bahamas, The',
    'BIH': 'Bosnia and Herzegovina',
    'BLR': 'Belarus',
    'BLZ': 'Belize',
    'BOL': 'Bolivia',
    'BRA': 'Brazil',
    'BRB': 'Barbados',
    'BTN': 'Bhutan',
    'BWA': 'Botswana',
    'CAF': 'Central African Republi

In [610]:
d = r.json()
d.keys()

dict_keys(['_meta', 'dataset', 'errors', 'provider', 'series'])

In [611]:
d['series']

{'docs': [{'dataset_code': 'ESY',
   'dataset_name': 'Enterprise Surveys',
   'dimensions': {'country': 'CHN',
    'frequency': 'A',
    'indicator': 'IC.FRM.BRIB.GRAFT2'},
   'indexed_at': '2022-07-01T19:18:23.311Z',
   'provider_code': 'WB',
   'series_code': 'A-IC.FRM.BRIB.GRAFT2-CHN',
   'series_name': 'Annual – Bribery depth (% of public transactions where a gift or informal payment was requested) – China'}],
 'limit': 1000,
 'num_found': 1,
 'offset': 0}

In [612]:
d['dataset'].keys()

dict_keys(['code', 'dimensions_codes_order', 'dimensions_values_labels', 'dir_hash', 'indexed_at', 'name', 'nb_series', 'provider_code', 'provider_name'])

In [613]:
d['dataset']['dimensions_values_labels'].keys()

dict_keys(['country', 'frequency', 'indicator'])

In [615]:
d['dataset']['dimensions_values_labels']

{'country': {'AFG': 'Afghanistan',
  'AGO': 'Angola',
  'ALB': 'Albania',
  'ARG': 'Argentina',
  'ARM': 'Armenia',
  'ATG': 'Antigua and Barbuda',
  'AZE': 'Azerbaijan',
  'BDI': 'Burundi',
  'BEL': 'Belgium',
  'BEN': 'Benin',
  'BFA': 'Burkina Faso',
  'BGD': 'Bangladesh',
  'BGR': 'Bulgaria',
  'BHS': 'Bahamas, The',
  'BIH': 'Bosnia and Herzegovina',
  'BLR': 'Belarus',
  'BLZ': 'Belize',
  'BOL': 'Bolivia',
  'BRA': 'Brazil',
  'BRB': 'Barbados',
  'BTN': 'Bhutan',
  'BWA': 'Botswana',
  'CAF': 'Central African Republic',
  'CHL': 'Chile',
  'CHN': 'China',
  'CIV': "Cote d'Ivoire",
  'CMR': 'Cameroon',
  'COD': 'Congo, Dem. Rep.',
  'COG': 'Congo, Rep.',
  'COL': 'Colombia',
  'CPV': 'Cabo Verde',
  'CRI': 'Costa Rica',
  'CYP': 'Cyprus',
  'CZE': 'Czech Republic',
  'DJI': 'Djibouti',
  'DMA': 'Dominica',
  'DOM': 'Dominican Republic',
  'ECU': 'Ecuador',
  'EGY': 'Egypt, Arab Rep.',
  'ERI': 'Eritrea',
  'EST': 'Estonia',
  'ETH': 'Ethiopia',
  'FJI': 'Fiji',
  'FSM': 'Microne

## 1.3 Build Indicators Catalog

In [ ]:
from src.metadata import build_indicator_catalog

indicators_catalog = build_indicator_catalog.get_indicator_catalog(datasets_df, client)
indicators_catalog

In [ ]:
import sdmx

In [ ]:
IMF_DATA = sdmx.Client('IMF_DATA')

In [ ]:
IMF_DATA.data('CPI')

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sdmx/rest/common.py:367: UserWarning: 'agency_id' argument is redundant for data queries
  getattr(self, f"handle_{query_type}")()
xml.Reader got no structure=… argument for StructureSpecificData


KeyboardInterrupt: 

In [ ]:
IMF_DATA.data

ValueError: can't determine a reader for response content type None

In [ ]:
data_msg = IMF_DATA.data('CPI', key='USA+CAN.CPI.CP01.IX.M', params={'startPeriod': 2018})
cpi_df = sdmx.to_pandas(data_msg)
cpi_df

xml.Reader got no structure=… argument for StructureSpecificData


TIME_PERIOD  INDEX_TYPE  COICOP_1999  TYPE_OF_TRANSFORMATION  COUNTRY  FREQUENCY  COMMON_REFERENCE_PERIOD  OVERLAP  SCALE  ACCESS_SHARING_LEVEL  SECURITY_CLASSIFICATION
2018-M01     CPI         CP01         IX                      CAN      M          2002A                    OL       0      PUBLIC_OPEN           PUB                        142.600000
2018-M02     CPI         CP01         IX                      CAN      M          2002A                    OL       0      PUBLIC_OPEN           PUB                        142.300000
2018-M03     CPI         CP01         IX                      CAN      M          2002A                    OL       0      PUBLIC_OPEN           PUB                        141.300000
2018-M04     CPI         CP01         IX                      CAN      M          2002A                    OL       0      PUBLIC_OPEN           PUB                        141.900000
2018-M05     CPI         CP01         IX                      CAN      M          2002A            

In [ ]:
from extraction import world_bank
client = world_bank.DBnomicsClient()

In [ ]:
BIS_datasets = client.list_datasets("BIS")

In [ ]:
provider_info = {}
provider_info[datasets['provider']['code']] = BIS_datasets['provider']['region'] + ' Region. ' + BIS_datasets['provider']['name']
provider_info

{'BIS': 'World Region. Bank for International Settlements'}

In [ ]:
BIS_datasets.keys()

dict_keys(['_meta', 'category_tree', 'provider'])

In [ ]:
datasets_BIS = {}

def get_datasets_pairs(l):
    datsets_descriptions = {}

    subname = ''
    for e in l:
        print(e['name'])
        subname = e['name']
        subsubname = ''
        for f in e['children']:
            subsubname = f['name']
            subsubsubname = ''
            for g in f['children']:
                subsubsubname = g['name']
                code = g['code']

                datsets_descriptions[code] = subname + subsubname + subsubsubname

    return datsets_descriptions

datasets_BIS_mapping = get_datasets_pairs(BIS_datasets['category_tree'])
datasets_BIS_mapping

BISWEB_CATSCHEME
        
Dataportal Category Scheme
        


{'WS_LBS_D_PUB': 'Dataportal Category Scheme\n        International banking\n          BIS locational banking',
 'WS_CBS_PUB': 'Dataportal Category Scheme\n        International banking\n          BIS consolidated banking',
 'WS_DEBT_SEC2_PUB': 'Dataportal Category Scheme\n        Debt securities\n          BIS international debt securities (BIS-compiled)',
 'WS_OTC_DERIV2': 'Dataportal Category Scheme\n        Derivatives\n          OTC derivatives outstanding',
 'WS_DER_OTC_TOV': 'Dataportal Category Scheme\n        Derivatives\n          OTC derivatives turnover',
 'WS_XTD_DERIV': 'Dataportal Category Scheme\n        Derivatives\n          Exchange traded derivatives',
 'WS_GLI': 'Dataportal Category Scheme\n        Global liquidity\n          Global liquidity indicators',
 'WS_TC': 'Dataportal Category Scheme\n        Credit\n          BIS long series on total credit',
 'WS_CREDIT_GAP': 'Dataportal Category Scheme\n        Credit\n          BIS credit-to-GDP gaps',
 'WS_DSR': 'Data

In [ ]:
len(datasets_BIS_mapping)

41

In [ ]:
client.list_series('BIS', 'WS_CBTA')

{'_meta': {'args': {'align_periods': False,
   'dataset_code': 'WS_CBTA',
   'dimensions': {},
   'facets': False,
   'format': 'json',
   'limit': 1000,
   'metadata': True,
   'observations': False,
   'offset': 0,
   'provider_code': 'BIS',
   'q': '',
   'series_code': None},
  'version': '22.1.17'},
 'dataset': {'attributes_labels': {'BREAKS': 'Breaks',
   'COLLECTION': 'Collection Indicator',
   'COLLECTION_DETAIL': 'Collection explanation detail',
   'COMMENT_DSET': '',
   'COMMENT_TS': 'Series comment',
   'COMPILING_ORG': 'Compiling agency',
   'CONF_STATUS': 'Confidentiality - status',
   'DATA_COMP': 'Data compilation',
   'DECIMALS': 'Decimals',
   'DISS_ORG': 'Data dissemination agency',
   'FISCAL_YEAR': 'Fiscal year',
   'METHOD_REF': 'Methodology reference',
   'OBS_PRE_BREAK': 'Observation pre-break value',
   'OBS_STATUS': 'Observation status',
   'SUPP_INFO_BREAKS': 'Supplemental information and breaks',
   'TIME_FORMAT': 'Time format',
   'UNIT_MULT': 'Unit multipli

In [ ]:
print('category_tree')
print(len(datasets['category_tree']))
for e in datasets['category_tree']:
        print('\t', e['name'])
        print('\t', e['code'], len(e['children']))
        for f in e['children']:
            print('\t\t', f['name'])
            print('\t\t', f['code'], len(f['children']))

            for g in f['children']:
                print('\t\t\t', g['name'])
                print('\t\t\t', g['code'])
                

category_tree
2
	 BISWEB_CATSCHEME
        
	 BISWEB_CATSCHEME 10
		 International banking
          
		 WEBSTATS_BANKING 2
			 BIS locational banking
			 WS_LBS_D_PUB
			 BIS consolidated banking
			 WS_CBS_PUB
		 Debt securities
          
		 WEBSTATS_SEC 1
			 BIS international debt securities (BIS-compiled)
			 WS_DEBT_SEC2_PUB
		 Derivatives
          
		 WEBSTATS_DER 3
			 OTC derivatives outstanding
			 WS_OTC_DERIV2
			 OTC derivatives turnover
			 WS_DER_OTC_TOV
			 Exchange traded derivatives
			 WS_XTD_DERIV
		 Global liquidity
          
		 WEBSTATS_GLI 1
			 Global liquidity indicators
			 WS_GLI
		 Total credit
          
		 WEBSTATS_TOTCRED 1
			 BIS long series on total credit
			 WS_TC
		 Credit gaps
          
		 WEBSTATS_CREDGAP 1
			 BIS credit-to-GDP gaps
			 WS_CREDIT_GAP
		 Debt service ratio
          
		 WEBSTATS_DSR 1
			 BIS debt service ratio
			 WS_DSR
		 Property prices
          
		 WEBSTATS_PP 1
			 Selected residential property prices
			 WS_SPP
		 Cons

In [ ]:
client.list_series('BIS', 'CBTA')

HTTPError: 404 Client Error: NOT FOUND for url: https://api.db.nomics.world/v22/series/BIS/CBTA?limit=1000&offset=0

In [ ]:
client.list_series('BIS', 'WS_CBTA')

{'_meta': {'args': {'align_periods': False,
   'dataset_code': 'WS_CBTA',
   'dimensions': {},
   'facets': False,
   'format': 'json',
   'limit': 1000,
   'metadata': True,
   'observations': False,
   'offset': 0,
   'provider_code': 'BIS',
   'q': '',
   'series_code': None},
  'version': '22.1.17'},
 'dataset': {'attributes_labels': {'BREAKS': 'Breaks',
   'COLLECTION': 'Collection Indicator',
   'COLLECTION_DETAIL': 'Collection explanation detail',
   'COMMENT_DSET': '',
   'COMMENT_TS': 'Series comment',
   'COMPILING_ORG': 'Compiling agency',
   'CONF_STATUS': 'Confidentiality - status',
   'DATA_COMP': 'Data compilation',
   'DECIMALS': 'Decimals',
   'DISS_ORG': 'Data dissemination agency',
   'FISCAL_YEAR': 'Fiscal year',
   'METHOD_REF': 'Methodology reference',
   'OBS_PRE_BREAK': 'Observation pre-break value',
   'OBS_STATUS': 'Observation status',
   'SUPP_INFO_BREAKS': 'Supplemental information and breaks',
   'TIME_FORMAT': 'Time format',
   'UNIT_MULT': 'Unit multipli

In [ ]:
def list_all_series(client, provider, dataset):
    all_series = []
    offset = 0
    limit = 1000

    while True:
        data = client.list_series(provider, dataset, limit, offset)
        series = data.get("series", [])
        if not series:
            break
        all_series.extend(series)
        offset += limit

    return all_series


In [ ]:
list_all_series(client, 'BIS', 'WS_CPMI_CT1')

HTTPError: 400 Client Error: BAD REQUEST for url: https://api.db.nomics.world/v22/series/BIS/WS_CPMI_CT1?limit=1000&offset=101000

# 2 - NEW

In [ ]:
from extraction import world_bank

client = world_bank.DBnomicsClient()

In [39]:
datasets_df = io.load_csv(config.METADATA_DIR / 'provider_datasets_selected.csv', index_col=0)
datasets_df.head(5)

,Provider Id,Provider Name,Provider Region,Dataset Id,Dataset Name,Subgroup 1 Id,Subgroup 1 Name,Subgroup 2 Id,Subgroup 2 Name,Country Param Name,Country Param Format,Frequency Param Name,Frequency Annual,Other Param Name
0,IMF,International Monetary Fund,World,BOP,Balance of Payments (BOP),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
2,IMF,International Monetary Fund,World,CPI,Consumer Price Index (CPI),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
3,IMF,International Monetary Fund,World,FDI,Financial Development Index,NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
4,IMF,International Monetary Fund,World,FM,Fiscal Monitor (FM),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR
5,IMF,International Monetary Fund,World,FSI,Financial Soundness Indicators (FSIs),NaN,NaN,NaN,NaN,REF_AREA,ISO2,FREQ,A,INDICATOR


In [40]:
providers = datasets_df['Provider Id'].unique()

dataset_params = {}
for i, dataset_row in datasets_df.iterrows():
    provider_id = dataset_row['Provider Id']
    dataset_id = dataset_row['Dataset Id']
    # print('Provider:', provider_id, 'Dataset:', dataset_id)

    if provider_id not in dataset_params.keys():
        dataset_params[provider_id] = {dataset_id: {}}
    else:
        dataset_params[provider_id][dataset_id] = {}

    try:
        dataset_dict = client.list_series(provider_id, dataset_id, limit=1)
        dataset_dim_dict = dataset_dict['dataset']['dimensions_values_labels']
        dataset_dim_dict.pop(dataset_row['Country Param Name'])
        dataset_dim_dict.pop(dataset_row['Frequency Param Name'])

        dataset_params[provider_id][dataset_id] = dataset_dim_dict

    except SyntaxError:
            print('Error querying dataset:', provider_id, dataset_id)

In [41]:
count_dataset_d = {}
count_provider_d = {}
count_d = {}
for provider_id, provider_d in dataset_params.items():
    count_d[provider_id] = {}
    count_dataset_d[provider_id] = {}
    v_provider = 0
    for dataset_id, dataset_d in provider_d.items():
        count_d[provider_id][dataset_id] = {}
        count_dataset_d[provider_id][dataset_id] = {}
        v_dataset = 1
        for param, param_d in dataset_d.items():
            v_dataset = v_dataset * len(param_d)
        
        count_dataset_d[provider_id][dataset_id] = v_dataset
        v_provider = v_provider + v_dataset
    count_provider_d[provider_id] = v_provider

count_provider_d

{'IMF': 18577, 'WB': 3266, 'BIS': 25944883500, 'UNCTAD': 69}

In [42]:
dataset_params['IMF']['BOP'].keys()

dict_keys(['INDICATOR'])

In [121]:
clean_d = {}

for provider, provider_d in dataset_params.items():
    clean_d[provider] = {}
    for dataset, dataset_d in provider_d.items():
        clean_d[provider][dataset] = {}
        
        for param, param_d in dataset_d.items():
            new_d = {}

            for ind, descr in sorted(param_d.items()):
                if ' ' in ind:
                    continue

                if provider == 'IMF' and param == 'INDICATOR':
                    if dataset in ['CPI', 'FDI', 'FM']:
                        new_d[ind] = descr
                    elif '_' in ind:
                        if ind[-3:] == 'USD' or ind[-2:] == 'PT':
                            if dataset != 'BOP' and len(ind.split('_')[0]) >= 3:
                                continue
                            else:
                                new_d[ind] = descr
                                
                        elif ind[-3:] == 'EUR' or ind[-3:] == 'XDC':
                            pass
                        else:
                            print('Weird Indicator', provider, dataset, ind)
                    else:
                        new_d[ind] = descr
                    # else:
                #     print('BAD')

                else:
                    new_d[ind] = descr

                clean_d[provider][dataset][param] = new_d

Weird Indicator IMF FSI All_Indicators
Weird Indicator IMF FSI All_Indicators_Minus_Discontinued
Weird Indicator IMF FSI Discontinued_Indicators
Weird Indicator IMF FSI ENDE_XDC_USD_RATE
Weird Indicator IMF FSI FSNA_NUM
Weird Indicator IMF FSI FSSH_BP
Weird Indicator IMF FSI FSSR_BP
Weird Indicator IMF FSI FS_ODX_ADU_MH
Weird Indicator IMF FSI FS_ODX_EXLN_NUM
Weird Indicator IMF FSI FS_ODX_HI_BP
Weird Indicator IMF FSI FS_ODX_HI_FSSH_BP
Weird Indicator IMF FSI FS_ODX_LDU_MH
Weird Indicator IMF FSI FS_ODX_LI_BP
Weird Indicator IMF FSI FS_ODX_LI_FSSH_BP
Weird Indicator IMF FSI FS_ODX_RD_BP
Weird Indicator IMF FSI FS_ODX_RD_FSSR_BP
Weird Indicator IMF FSI FS_ODX_RL_BP
Weird Indicator IMF FSI FS_ODX_RL_FSSR_BP
Weird Indicator IMF GENDER_EQUALITY All_Indicators
Weird Indicator IMF GENDER_EQUALITY GE_GDI
Weird Indicator IMF GENDER_EQUALITY GE_GII
Weird Indicator IMF HPDD GGXWDG_GDP
Weird Indicator IMF IFS 1C_ALL_INDICATORS
Weird Indicator IMF IFS 26J___XDR
Weird Indicator IMF IFS AIPCO_IX
We

In [122]:
len(dataset_params['IMF']['BOP']['INDICATOR']), len(clean_d['IMF']['BOP']['INDICATOR'])

(6654, 2218)

In [123]:
clean_d['IMF']['CPI']['INDICATOR'] = {
    # 'PCPIA_IX': 'Clothing and footwear',
    # 'PCPIEC_PC_CP_A_PT': 'Communication, Percentage change, Previous year',
    # 'PCPIED_PC_CP_A_PT': 'Education, Percentage change, Previous year',
    'PCPI_PC_CP_A_PT': 'Consumer Price Index, All items, Percentage change, Previous year'
}
len(dataset_params['IMF']['CPI']['INDICATOR']), len(clean_d['IMF']['CPI']['INDICATOR'])

(178, 1)

In [124]:
dataset_params['IMF'].keys()

dict_keys(['BOP', 'CPI', 'FDI', 'FM', 'FSI', 'GENDER_EQUALITY', 'GFSMAB', 'HPDD', 'IFS', 'IRFCL', 'MFS', 'PGCS', 'PSBSFAD', 'WoRLD'])

In [125]:
len(dataset_params['IMF']['FSI']['INDICATOR']), len(clean_d['IMF']['FSI']['INDICATOR'])

(2453, 669)

In [126]:
len(dataset_params['IMF']['FDI']['INDICATOR']), len(clean_d['IMF']['FDI']['INDICATOR'])

(9, 9)

In [127]:
len(dataset_params['IMF']['CPI']['INDICATOR']), len(clean_d['IMF']['CPI']['INDICATOR'])

(178, 1)

In [128]:
len(dataset_params['IMF']['FSI']['INDICATOR']), len(clean_d['IMF']['FSI']['INDICATOR'])

(2453, 669)

In [129]:
count_dataset_d = {}
count_provider_d = {}
count_d = {}
for provider_id, provider_d in clean_d.items():
    count_d[provider_id] = {}
    count_dataset_d[provider_id] = {}
    v_provider = 0
    for dataset_id, dataset_d in provider_d.items():
        count_d[provider_id][dataset_id] = {}
        count_dataset_d[provider_id][dataset_id] = {}
        v_dataset = 1
        for param, param_d in dataset_d.items():
            v_dataset = v_dataset * len(param_d)
        
        count_dataset_d[provider_id][dataset_id] = v_dataset
        v_provider = v_provider + v_dataset
    count_provider_d[provider_id] = v_provider

count_provider_d

{'IMF': 4346, 'WB': 3266, 'BIS': 25944883500, 'UNCTAD': 69}

In [130]:
count_dataset_d['IMF']

{'BOP': 2218,
 'CPI': 1,
 'FDI': 9,
 'FM': 9,
 'FSI': 669,
 'GENDER_EQUALITY': 0,
 'GFSMAB': 560,
 'HPDD': 0,
 'IFS': 30,
 'IRFCL': 0,
 'MFS': 15,
 'PGCS': 4,
 'PSBSFAD': 831,
 'WoRLD': 0}

In [36]:
bop_indicators_d = dataset_params['IMF']['BOP']['INDICATOR']
format_indicators_d = {}

ind_template = {
    'DESCRIPTION' : '',
    'VERSION' : {},
    # 'SUBCATEGORY' : {},
    'CURRENCY': {},
    'OTHER': {}
}

for ind, descr in sorted(bop_indicators_d.items()):
    ind_sections = ind.split('_')
    descr_sections = descr.split(',')
    ind_id = ind_sections[0]
    ind_descr = ','.join(descr_sections[:-1])

    # if ind_id != 'BACK':
    #     continue

    # ind_sub = ''
    # for i in range(i, len(ind_id)):
    #     if ind_id[0:i] in format_indicators_d.keys():
    #         ind_sub = ind_id[i:]
    #         ind_id = ind_id[0:i]

    #         print('Sub Indicator:', ind_id, ind_sub)
    #         break
    #     else:
    #         continue

    if not ind_id in format_indicators_d.keys():
        format_indicators_d[ind_id] = ind_template

    # if ind_sub != '':
    desrc_ind = format_indicators_d[ind_id]['DESCRIPTION']
    descr_sub = ','.join(ind_descr).replace(desrc_ind, '')
    # format_indicators_d[ind_id]['SUBCATEGORY'][ind_sub] = descr_sub
    # else:

    if ind_descr == 'Total' and ind_id == 'BACK':
        print(ind, descr)
    format_indicators_d[ind_id]['DESCRIPTION'] = ind_descr

    if len(ind_sections) > 2:
        # print(ind_sections)
        for i in range(1, len(ind_sections)-1):
            ind_sect = ind_sections[i]

            if ind_sect == 'BP6':
                format_indicators_d[ind_id]['VERSION'][ind_sect] = ind_sect
            else:
                format_indicators_d[ind_id]['OTHER'][ind_sect] = [ind_sect]

    format_indicators_d[ind_id]['CURRENCY'][ind_sections[-1]] = descr_sections[-1]

    # break

format_indicators_d


{'BACK': {'DESCRIPTION': 'Total',
  'VERSION': {'BP6': 'BP6'},
  'CURRENCY': {'EUR': ' Euro',
   'USD': ' US Dollars',
   'XDC': ' National Currency'},
  'OTHER': {'SPE': ['SPE'],
   'MLT': ['MLT'],
   'S': ['S'],
   'L': ['L'],
   'CD': ['CD'],
   'DB': ['DB'],
   'NRES': ['NRES'],
   '1YOL': ['1YOL'],
   'CB': ['CB'],
   'FX': ['FX'],
   'EUR': ['EUR'],
   'JPY': ['JPY'],
   'OTH': ['OTH'],
   'USD': ['USD'],
   'NC': ['NC'],
   'UA': ['UA'],
   'GG': ['GG'],
   'ICL': ['ICL'],
   'ODC': ['ODC'],
   'OTSOFC': ['OTSOFC'],
   'OTSOTH': ['OTSOTH'],
   'OTS': ['OTS'],
   'NV': ['NV'],
   'I': ['I'],
   'SDR': ['SDR'],
   'X': ['X'],
   'CL': ['CL'],
   'DS': ['DS'],
   'D': ['D'],
   'OL': ['OL'],
   'O': ['O'],
   'RL': ['RL']}},
 'BAXEF': {'DESCRIPTION': 'Total',
  'VERSION': {'BP6': 'BP6'},
  'CURRENCY': {'EUR': ' Euro',
   'USD': ' US Dollars',
   'XDC': ' National Currency'},
  'OTHER': {'SPE': ['SPE'],
   'MLT': ['MLT'],
   'S': ['S'],
   'L': ['L'],
   'CD': ['CD'],
   'DB': ['DB'

In [37]:
format_indicators_d['BACK']

{'DESCRIPTION': 'Total',
 'VERSION': {'BP6': 'BP6'},
 'CURRENCY': {'EUR': ' Euro',
  'USD': ' US Dollars',
  'XDC': ' National Currency'},
 'OTHER': {'SPE': ['SPE'],
  'MLT': ['MLT'],
  'S': ['S'],
  'L': ['L'],
  'CD': ['CD'],
  'DB': ['DB'],
  'NRES': ['NRES'],
  '1YOL': ['1YOL'],
  'CB': ['CB'],
  'FX': ['FX'],
  'EUR': ['EUR'],
  'JPY': ['JPY'],
  'OTH': ['OTH'],
  'USD': ['USD'],
  'NC': ['NC'],
  'UA': ['UA'],
  'GG': ['GG'],
  'ICL': ['ICL'],
  'ODC': ['ODC'],
  'OTSOFC': ['OTSOFC'],
  'OTSOTH': ['OTSOTH'],
  'OTS': ['OTS'],
  'NV': ['NV'],
  'I': ['I'],
  'SDR': ['SDR'],
  'X': ['X'],
  'CL': ['CL'],
  'DS': ['DS'],
  'D': ['D'],
  'OL': ['OL'],
  'O': ['O'],
  'RL': ['RL']}}

In [ ]:
new_d = {}
for provider_id, provider_d in dataset_params.items():
    new_d[provider_id] = {}
    for dataset_id, dataset_d in provider_d.items():
        new_d[provider_id][dataset_id] = {}
        for param, param_d in dataset_d.items():
            new_d[provider_id][dataset_id][param] = {}

            new_param_d = {}
            for k, v in param_d.items():
                if (k[-4:] == '_EUR' or k[-4:] == '_XDC') and (k[:-4] + '_USD') in param_d.keys():
                    pass
                else:
                    new_param_d[k] = v

            new_d[provider_id][dataset_id][param] = new_param_d

In [ ]:
count_dataset_d = {}
count_d = {}
for provider_id, provider_d in new_d.items():
    count_d[provider_id] = {}
    count_dataset_d[provider_id] = {}
    for dataset_id, dataset_d in provider_d.items():
        count_d[provider_id][dataset_id] = {}
        count_dataset_d[provider_id][dataset_id] = {}
        v = 1
        for param, param_d in dataset_d.items():
            v = v * len(new_d[provider_id][dataset_id][param])
        
        count_dataset_d[provider_id][dataset_id] = v

In [ ]:
import json

json_str = json.dumps(new_d, indent=4)
with open(config.METADATA_DIR / 'datasets_dimensions.json', 'w') as f:
    f.write(json_str)

In [ ]:
count_dataset_d = {}
count_d = {}
for provider_id, provider_d in new_d.items():
    count_d[provider_id] = {}
    count_dataset_d[provider_id] = {}
    for dataset_id, dataset_d in provider_d.items():
        count_d[provider_id][dataset_id] = {}
        count_dataset_d[provider_id][dataset_id] = {}
        v = 1
        for param, param_d in dataset_d.items():
            v = v * len(new_d[provider_id][dataset_id][param])
        
        count_dataset_d[provider_id][dataset_id] = v

In [ ]:
count_d

{'IMF': {'BOP': {'INDICATOR': 2218},
  'CPI': {'INDICATOR': 178},
  'FDI': {'INDICATOR': 9},
  'FM': {'INDICATOR': 9},
  'FSI': {'INDICATOR': 959},
  'GENDER_EQUALITY': {'INDICATOR': 3},
  'GFSMAB': {'CLASSIFICATION': 35, 'REF_SECTOR': 8, 'UNIT_MEASURE': 2},
  'HPDD': {'INDICATOR': 1},
  'IFS': {'INDICATOR': 1301},
  'IRFCL': {'INDICATOR': 295, 'REF_SECTOR': 5},
  'MFS': {'INDICATOR': 575},
  'PGCS': {'INDICATOR': 15},
  'PSBSFAD': {'INDICATOR': 1235},
  'WoRLD': {'INDICATOR': 15}},
 'WB': {'DBS': {'indicator': 194},
  'ESY': {'indicator': 115},
  'FDX': {'indicator': 1226},
  'GEM': {'indicator': 36},
  'GEP': {'indicator': 1},
  'JOB': {'indicator': 166},
  'WDI': {'indicator': 1492},
  'WGI': {'indicator': 36}},
 'BIS': {'WS_GLI': {'BORROWERS_SECTOR': 20,
   'CURR_DENOM': 324,
   'LENDERS_SECTOR': 20,
   'L_INSTR': 23,
   'L_POS_TYPE': 8,
   'UNIT_MEASURE': 1088},
  'WS_CREDIT_GAP': {'CG_DTYPE': 3, 'TC_BORROWERS': 5, 'TC_LENDERS': 20}},
 'UNCTAD': {'MTTASA': {'flow': 2, 'measure': 2

In [ ]:
count_dataset_d

{'IMF': {'BOP': 2218,
  'CPI': 178,
  'FDI': 9,
  'FM': 9,
  'FSI': 959,
  'GENDER_EQUALITY': 3,
  'GFSMAB': 560,
  'HPDD': 1,
  'IFS': 1301,
  'IRFCL': 1475,
  'MFS': 575,
  'PGCS': 15,
  'PSBSFAD': 1235,
  'WoRLD': 15},
 'WB': {'DBS': 194,
  'ESY': 115,
  'FDX': 1226,
  'GEM': 36,
  'GEP': 1,
  'JOB': 166,
  'WDI': 1492,
  'WGI': 36},
 'BIS': {'WS_GLI': 25944883200, 'WS_CREDIT_GAP': 300},
 'UNCTAD': {'MTTASA': 4,
  'MTBA': 2,
  'MTTGRA': 2,
  'MPCADIOEAIA': 6,
  'GASBTBIA': 8,
  'GASBTOIA': 16,
  'FDIIAOFASA': 20,
  'GDPTAPCCAC2PA': 4,
  'RGDPTAPCGRA': 2,
  'NGTAPCA': 2,
  'TAUPA': 2,
  'TPGRA': 1},
 'ILO': {'EAR_GGAP_OCU_RT': 4350,
  'SDG_1041_NOC_RT': 199,
  'EES_TEES_ECO_OCU_NB': 386512,
  'PSE_TPSE_GOV_NB': 2040,
  'GDP_205U_NOC_NB': 274,
  'SDG_B821_NOC_RT': 199,
  'INJ_FATL_ECO_NB': 12505}}

In [ ]:
with open(config.METADATA_DIR / 'datasets_dimensions.json', 'r') as f:
    d = json.load(f)

print(d)

{'IMF': {'BOP': {'INDICATOR': {'BACK_BP6_USD': 'Net Lending (+) / Net Borrowing (-) (Balance from Current and Capital Account), US Dollars', 'BAXEF_BP6_USD': 'Supplementary Items, Arrears not in Exceptional Financing, US Dollars', 'BCAXF_BP6_USD': 'Supplementary Items, Current Account, Net (Excluding Exceptional Financing), US Dollars', 'BCA_BP6_SPE_USD': 'Current Account, Total, Net, Special Purpose Entities, US Dollar', 'BCA_BP6_USD': 'Current Account, Total, Net, US Dollars', 'BEFDDAAI_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Original Interest/Coupon, US Dollars', 'BEFDDAAPI_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Penalty Interest, US Dollars', 'BEFDDAAP_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arrears, Principal, US Dollars', 'BEFDDAA_BP6_USD': 'Exceptional Financing, Direct Investment, Debt Instruments, Accumulations of Arr

In [ ]:
d['BIS']['WS_GLI']

{'BORROWERS_SECTOR': {'A': 'All sectors',
  'B': 'Banks, total',
  'G': 'General government',
  'H': 'Households and NPISHs',
  'N': 'Non-banks, total'},
 'CURR_DENOM': {'USD': 'US dollar'},
 'LENDERS_SECTOR': {'A': 'All sectors',
  'B': 'Banks, total',
  'G': 'General government',
  'H': 'Households and NPISHs',
  'N': 'Non-banks, total'},
 'L_INSTR': {'A': 'All instruments',
  'B': 'Credit (loans & debt securities)',
  'D': 'Debt securities',
  'E': 'Risk-weighted assets',
  'G': 'Loans and deposits',
  'X': 'Equity instruments',
  'Y': 'Residual instrument'},
 'L_POS_TYPE': {'A': 'All'},
 'UNIT_MEASURE': {'770': 'Percentage of GDP'}}

In [ ]:
count_dataset_d = {}
count_provider_d = {}
count_d = {}
for provider_id, provider_d in d.items():
    count_d[provider_id] = {}
    count_dataset_d[provider_id] = {}
    v_provider = 0
    for dataset_id, dataset_d in provider_d.items():
        count_d[provider_id][dataset_id] = {}
        count_dataset_d[provider_id][dataset_id] = {}
        v_dataset = 1
        for param, param_d in dataset_d.items():
            v_dataset = v_dataset * len(param_d)
        
        count_dataset_d[provider_id][dataset_id] = v_dataset
        v_provider = v_provider + v_dataset
    count_provider_d[provider_id] = v_provider

count_provider_d

{'IMF': 8547, 'WB': 3266, 'BIS': 250, 'UNCTAD': 69}

In [ ]:
count_dataset_d

{'IMF': {'BOP': 2218,
  'CPI': 178,
  'FDI': 9,
  'FM': 9,
  'FSI': 959,
  'GENDER_EQUALITY': 3,
  'GFSMAB': 560,
  'HPDD': 1,
  'IFS': 1301,
  'IRFCL': 1475,
  'MFS': 575,
  'PGCS': 9,
  'PSBSFAD': 1235,
  'WoRLD': 15},
 'WB': {'DBS': 194,
  'ESY': 115,
  'FDX': 1226,
  'GEM': 36,
  'GEP': 1,
  'JOB': 166,
  'WDI': 1492,
  'WGI': 36},
 'BIS': {'WS_GLI': 175, 'WS_CREDIT_GAP': 75},
 'UNCTAD': {'MTTASA': 4,
  'MTBA': 2,
  'MTTGRA': 2,
  'MPCADIOEAIA': 6,
  'GASBTBIA': 8,
  'GASBTOIA': 16,
  'FDIIAOFASA': 20,
  'GDPTAPCCAC2PA': 4,
  'RGDPTAPCGRA': 2,
  'NGTAPCA': 2,
  'TAUPA': 2,
  'TPGRA': 1}}

In [ ]:
d = client.list_series('ILO', 'LAP_2FTM_NOC_RT', limit=1)
d

{'_meta': {'args': {'align_periods': False,
   'dataset_code': 'LAP_2FTM_NOC_RT',
   'dimensions': {},
   'facets': False,
   'format': 'json',
   'limit': 1,
   'metadata': True,
   'observations': False,
   'offset': 0,
   'provider_code': 'ILO',
   'q': '',
   'series_code': None},
  'version': '22.1.17'},
 'dataset': {'attributes_labels': {'OBSV_STATUS': 'Observation status'},
  'attributes_values_labels': {'OBSV_STATUS': {'A': 'Adjusted',
    'B': 'Break in series',
    'C': 'Confidential',
    'E': 'Estimate',
    'I': 'Imputation',
    'M': 'Model-based extrapolation',
    'P': 'Provisional',
    'R': 'Real value',
    'S': 'Not significant',
    'U': 'Unreliable'}},
  'code': 'LAP_2FTM_NOC_RT',
  'dimensions_codes_order': ['ref_area', 'source', 'frequency'],
  'dimensions_labels': {'frequency': 'Frequency',
   'ref_area': 'Reference area',
   'source': 'Source'},
  'dimensions_values_labels': {'frequency': {'A': 'Annual'},
   'ref_area': {'X01': 'World',
    'X02': 'World: Low 

In [ ]:
d.keys()

dict_keys(['_meta', 'dataset', 'errors', 'provider', 'series'])

In [ ]:
d['dataset'].keys()

dict_keys(['attributes_labels', 'attributes_values_labels', 'code', 'dimensions_codes_order', 'dimensions_labels', 'dimensions_values_labels', 'dir_hash', 'indexed_at', 'name', 'nb_series', 'provider_code', 'provider_name'])

In [ ]:
d['dataset']['dimensions_values_labels'].keys()

dict_keys(['frequency', 'ref_area', 'source'])

In [ ]:
d['dataset']['dimensions_values_labels']['ref_area']

{'X01': 'World',
 'X02': 'World: Low income',
 'X03': 'World: Lower-middle income',
 'X04': 'World: Upper-middle income',
 'X05': 'World: High income',
 'X06': 'Africa',
 'X07': 'Africa: Low income',
 'X08': 'Africa: Lower-middle income',
 'X09': 'Africa: Upper-middle income',
 'X10': 'Northern Africa',
 'X11': 'Northern Africa: Lower-middle income',
 'X13': 'Sub-Saharan Africa',
 'X14': 'Sub-Saharan Africa: Low income',
 'X15': 'Sub-Saharan Africa: Lower-middle income',
 'X16': 'Sub-Saharan Africa: Upper-middle income',
 'X17': 'Central Africa',
 'X18': 'Eastern Africa',
 'X19': 'Southern Africa',
 'X20': 'Western Africa',
 'X21': 'Americas',
 'X23': 'Americas: Lower-middle income',
 'X24': 'Americas: Upper-middle income',
 'X25': 'Americas: High income',
 'X26': 'Latin America and the Caribbean',
 'X28': 'Latin America and the Caribbean: Lower-middle income',
 'X29': 'Latin America and the Caribbean: Upper-middle income',
 'X30': 'Latin America and the Caribbean: High income',
 'X31'

In [ ]:
import requests

url = "https://api.db.nomics.world/v22" + "/series/WB/ESY/?"

r = requests.get(url)
r.raise_for_status()
r.json()

{'_meta': {'args': {'align_periods': False,
   'dataset_code': 'ESY',
   'dimensions': {},
   'facets': False,
   'format': 'json',
   'limit': 1000,
   'metadata': True,
   'observations': False,
   'offset': 0,
   'provider_code': 'WB',
   'q': '',
   'series_code': None},
  'version': '22.1.17'},
 'dataset': {'code': 'ESY',
  'dimensions_codes_order': ['frequency', 'indicator', 'country'],
  'dimensions_values_labels': {'country': {'AFG': 'Afghanistan',
    'AGO': 'Angola',
    'ALB': 'Albania',
    'ARG': 'Argentina',
    'ARM': 'Armenia',
    'ATG': 'Antigua and Barbuda',
    'AZE': 'Azerbaijan',
    'BDI': 'Burundi',
    'BEL': 'Belgium',
    'BEN': 'Benin',
    'BFA': 'Burkina Faso',
    'BGD': 'Bangladesh',
    'BGR': 'Bulgaria',
    'BHS': 'Bahamas, The',
    'BIH': 'Bosnia and Herzegovina',
    'BLR': 'Belarus',
    'BLZ': 'Belize',
    'BOL': 'Bolivia',
    'BRA': 'Brazil',
    'BRB': 'Barbados',
    'BTN': 'Bhutan',
    'BWA': 'Botswana',
    'CAF': 'Central African Republi

In [ ]:
d = r.json()
d.keys()

dict_keys(['_meta', 'dataset', 'errors', 'provider', 'series'])

In [ ]:
d['series']

{'docs': [{'dataset_code': 'ESY',
   'dataset_name': 'Enterprise Surveys',
   'dimensions': {'country': 'CHN',
    'frequency': 'A',
    'indicator': 'IC.FRM.BRIB.GRAFT2'},
   'indexed_at': '2022-07-01T19:18:23.311Z',
   'provider_code': 'WB',
   'series_code': 'A-IC.FRM.BRIB.GRAFT2-CHN',
   'series_name': 'Annual – Bribery depth (% of public transactions where a gift or informal payment was requested) – China'}],
 'limit': 1000,
 'num_found': 1,
 'offset': 0}

In [ ]:
d['dataset'].keys()

dict_keys(['code', 'dimensions_codes_order', 'dimensions_values_labels', 'dir_hash', 'indexed_at', 'name', 'nb_series', 'provider_code', 'provider_name'])

In [ ]:
d['dataset']['dimensions_values_labels'].keys()

dict_keys(['country', 'frequency', 'indicator'])

In [ ]:
d['dataset']['dimensions_values_labels']

{'country': {'AFG': 'Afghanistan',
  'AGO': 'Angola',
  'ALB': 'Albania',
  'ARG': 'Argentina',
  'ARM': 'Armenia',
  'ATG': 'Antigua and Barbuda',
  'AZE': 'Azerbaijan',
  'BDI': 'Burundi',
  'BEL': 'Belgium',
  'BEN': 'Benin',
  'BFA': 'Burkina Faso',
  'BGD': 'Bangladesh',
  'BGR': 'Bulgaria',
  'BHS': 'Bahamas, The',
  'BIH': 'Bosnia and Herzegovina',
  'BLR': 'Belarus',
  'BLZ': 'Belize',
  'BOL': 'Bolivia',
  'BRA': 'Brazil',
  'BRB': 'Barbados',
  'BTN': 'Bhutan',
  'BWA': 'Botswana',
  'CAF': 'Central African Republic',
  'CHL': 'Chile',
  'CHN': 'China',
  'CIV': "Cote d'Ivoire",
  'CMR': 'Cameroon',
  'COD': 'Congo, Dem. Rep.',
  'COG': 'Congo, Rep.',
  'COL': 'Colombia',
  'CPV': 'Cabo Verde',
  'CRI': 'Costa Rica',
  'CYP': 'Cyprus',
  'CZE': 'Czech Republic',
  'DJI': 'Djibouti',
  'DMA': 'Dominica',
  'DOM': 'Dominican Republic',
  'ECU': 'Ecuador',
  'EGY': 'Egypt, Arab Rep.',
  'ERI': 'Eritrea',
  'EST': 'Estonia',
  'ETH': 'Ethiopia',
  'FJI': 'Fiji',
  'FSM': 'Microne